In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

In [ ]:
df=pd.read_csv("/content/hour.csv")

In [ ]:
df_clean = df.drop(columns=['instant', 'dteday', 'casual', 'registered', 'atemp'])

In [ ]:
df_clean['hr_sin'] = np.sin(2 * np.pi * df_clean['hr'] / 24.0)
df_clean['hr_cos'] = np.cos(2 * np.pi * df_clean['hr'] / 24.0)
df_clean['mnth_sin'] = np.sin(2 * np.pi * df_clean['mnth'] / 12.0)
df_clean['mnth_cos'] = np.cos(2 * np.pi * df_clean['mnth'] / 12.0)
df_clean = df_clean.drop(columns=['hr', 'mnth'])

In [ ]:
cols_to_encode = ['season', 'weathersit', 'weekday']
df_cleaned = pd.get_dummies(df_clean, columns=cols_to_encode, drop_first=True)

In [ ]:
X = df_cleaned.drop(columns=['cnt'])
Y = df_cleaned['cnt']
print(f"Feature matrix (X) shape: {X.shape}")
print(f"Target vector (y) shape:  {Y.shape}\n")

Feature matrix (X) shape: (17379, 22)
Target vector (y) shape:  (17379,)



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}\n")

X_train shape: (13903, 22) | y_train shape: (13903,)
X_test shape:  (3476, 22)  | y_test shape:  (3476,)



In [ ]:
continuous_cols = ['temp', 'hum', 'windspeed']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

LinearRegression()

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
y_pred= np.clip(y_pred, a_min=0, a_max=None)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Evaluation Metrics ---")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE):     {mae:.2f}")
print(f"R-squared Score (R²):           {r2:.4f}\n")

--- Model Evaluation Metrics ---
Root Mean Squared Error (RMSE): 123.84
Mean Absolute Error (MAE):     88.81
R-squared Score (R²):           0.5157



In [ ]:
joblib.dump(model, 'bike_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=120)

# ---------------------------------------------------------
# Plot 1: Actual vs. Predicted Scatter Plot
# ---------------------------------------------------------
sns.scatterplot(x=y_test, y=y_pred, ax=axes[0], alpha=0.4, color='#2b5c8f')

# Perfect prediction reference line (y = x)
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Ideal Prediction (y = x)')

axes[0].set_title('Actual vs. Predicted Demand', fontsize=13, pad=10)
axes[0].set_xlabel('Actual Demand (y_test)', fontsize=11)
axes[0].set_ylabel('Predicted Demand (y_pred)', fontsize=11)
axes[0].legend(loc='upper left')

# ---------------------------------------------------------
# Plot 2: Residuals Plot (Errors)
# ---------------------------------------------------------
residuals = y_test - y_pred
sns.scatterplot(x=y_pred, y=residuals, ax=axes[1], alpha=0.4, color='#8e44ad')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)

axes[1].set_title('Residuals (Actual - Predicted)', fontsize=13, pad=10)
axes[1].set_xlabel('Predicted Demand (y_pred)', fontsize=11)
axes[1].set_ylabel('Error / Residual (cnt)', fontsize=11)

# ---------------------------------------------------------
# Plot 3: Time-Series Sample Overlay (First 100 Hours)
# ---------------------------------------------------------
sample_n = 100
sample_df = pd.DataFrame({
    'Actual': y_test.values[:sample_n],
    'Predicted': y_pred[:sample_n]
}).reset_index()

sns.lineplot(data=sample_df, x='index', y='Actual', ax=axes[2], label='Actual', color='#1f77b4', linewidth=2)
sns.lineplot(data=sample_df, x='index', y='Predicted', ax=axes[2], label='Predicted', color='#ff7f0e', linestyle='--', linewidth=2)

axes[2].set_title('First 100 Hours Overlay', fontsize=13, pad=10)
axes[2].set_xlabel('Sample Index (Hours)', fontsize=11)
axes[2].set_ylabel('Bike Rental Count (cnt)', fontsize=11)
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.savefig('model_error_analysis2.png')
plt.show()